# Integración numérica — rectángulos y trapecio

Copia la función del método elegido junto a los imports. `N` siempre cuenta subintervalos; `N+1` cuenta nodos. Los métodos aceptan funciones que reciben un escalar.

**Fuente:** `04-integrales, pp. 4–10` y notebook de clase original. Revisión de estudio con IA; usar en evaluación solo si el material asistido está autorizado.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Métodos para reutilizar
Rectángulo izquierdo: $h\sum_{i=0}^{N-1} f(x_i)$. Trapecio: $h[f(a)/2+\sum_{i=1}^{N-1}f(x_i)+f(b)/2]$. El punto medio es una variante complementaria.

In [2]:
def rectangulo(f, a, b, N):
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError('N debe ser un entero positivo')
    h = (b-a)/N
    x = np.linspace(a, b, N+1)
    return h*sum(f(xi) for xi in x[:-1])

def punto_medio(f, a, b, N):
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError('N debe ser un entero positivo')
    h = (b-a)/N
    x = np.linspace(a, b, N+1)
    return h*sum(f(xi) for xi in (x[:-1]+x[1:])/2)

def trapezoide(f, a, b, N):
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError('N debe ser un entero positivo')
    h = (b-a)/N
    x = np.linspace(a, b, N+1)
    return h*(f(x[0])/2 + sum(f(xi) for xi in x[1:-1]) + f(x[-1])/2)

## 2. Ejemplo: modificar únicamente f, a, b y N
Ejemplo original de clase: integrar $x^3$ entre 1 y 2.

In [3]:
def f(x):
    return x**3
a, b, N = 1, 2, 100
exacta = (b**4-a**4)/4
for metodo in (rectangulo, punto_medio, trapezoide):
    I = metodo(f, a, b, N)
    print(metodo.__name__, I, 'error:', abs(I-exacta))
assert np.isclose(trapezoide(lambda x: 2*x+1, 0, 2, 7), 6)
assert np.isclose(punto_medio(lambda x: 2*x+1, 0, 2, 7), 6)

rectangulo 3.7150749999999992 error: 0.03492500000000076
punto_medio 3.7499625000000005 error: 3.7499999999468514e-05
trapezoide 3.7500749999999994 error: 7.499999999938112e-05


## 3. Si ya tienes los datos (x, y)
No necesitas reconstruir f. Esta versión del trapecio admite intervalos de ancho distinto, en orden creciente o decreciente.

In [4]:
def trapezoide_datos(x, y):
    x, y = np.asarray(x), np.asarray(y)
    if x.ndim != 1 or y.ndim != 1 or len(x) != len(y) or len(x) < 2:
        raise ValueError('x e y deben ser vectores de igual longitud >= 2')
    dx = np.diff(x)
    if not (np.all(dx > 0) or np.all(dx < 0)):
        raise ValueError('Los nodos deben ser estrictamente monótonos')
    return np.sum(dx*(y[:-1]+y[1:])/2)

x = np.array([0., .2, 1., 2.])
y = 2*x+1
print(trapezoide_datos(x,y))
assert np.isclose(trapezoide_datos(x,y),6)

6.0


## 4. Variante NumPy y consejos
Si `f` acepta un arreglo NumPy, evaluar `f(x)` de una vez evita el ciclo Python. `np.vectorize` por sí solo no acelera el cálculo. Para funciones suaves, los errores globales son O(h) para rectángulo izquierdo y O(h²) para punto medio y trapecio.

In [5]:
x = np.linspace(a,b,N+1)
y = f(x)
print(trapezoide_datos(x,y))
assert np.isclose(trapezoide_datos(x,y),trapezoide(f,a,b,N))

3.7500750000000003


**Cuidado:** `np.arange(a,b,h)` con flotantes puede producir una cantidad inesperada de puntos; `linspace` fija la cantidad. Si inviertes los límites cambia el signo. Si hay discontinuidades o singularidades, separa el intervalo o cambia de método. Para comparar precisión, duplica N y compara los resultados; esa diferencia no garantiza por sí sola el error real.